In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent

INTERACTIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "electronics_interactions"
)

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [2]:
query = f"""
SELECT COUNT(*) AS interaction_count
FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
"""

result = con.execute(query).df()

result

,interaction_count
0,43365426


In [3]:
query = f"""
SELECT
    EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) AS year,
    COUNT(*) AS interactions
FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
GROUP BY year
ORDER BY year
"""

year_counts = con.execute(query).df()

year_counts

,year,interactions
0,1996,1
1,1998,8
2,1999,410
3,2000,3694
4,2001,6747
5,2002,8981
6,2003,12162
7,2004,17398
8,2005,32642
9,2006,53547


In [4]:
query = f"""
SELECT
    CASE
        WHEN EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) <= 2021
            THEN 'train'
        WHEN EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) = 2022
            THEN 'validation'
        WHEN EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) = 2023
            THEN 'test'
        ELSE 'outside'
    END AS split,
    COUNT(*) AS interactions
FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
GROUP BY split
ORDER BY split
"""

split_counts = con.execute(query).df()

split_counts

,split,interactions
0,test,1888698
1,train,37006469
2,validation,4470259


In [5]:
query = f"""
WITH item_history AS (
    SELECT
        parent_asin,
        COUNT(*) AS train_interactions
    FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
    WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) <= 2021
    GROUP BY parent_asin
)

SELECT
    CASE
        WHEN train_interactions = 1 THEN '1'
        WHEN train_interactions BETWEEN 2 AND 5 THEN '2-5'
        WHEN train_interactions BETWEEN 6 AND 20 THEN '6-20'
        WHEN train_interactions >= 21 THEN '21+'
    END AS history_bucket,
    COUNT(*) AS item_count
FROM item_history
GROUP BY history_bucket
ORDER BY
    CASE history_bucket
        WHEN '1' THEN 1
        WHEN '2-5' THEN 2
        WHEN '6-20' THEN 3
        WHEN '21+' THEN 4
    END
"""

train_item_buckets = con.execute(query).df()

train_item_buckets

,history_bucket,item_count
0,1,473594
1,2-5,443218
2,6-20,276219
3,21+,201904


In [6]:
query = f"""
WITH train_history AS (
    SELECT
        parent_asin,
        COUNT(*) AS train_interactions
    FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
    WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) <= 2021
    GROUP BY parent_asin
),

test_items AS (
    SELECT
        DISTINCT parent_asin
    FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
    WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) = 2023
)

SELECT
    CASE
        WHEN COALESCE(train_interactions, 0) = 0
            THEN 'true_cold'
        WHEN train_interactions BETWEEN 1 AND 5
            THEN 'low_history'
        WHEN train_interactions BETWEEN 6 AND 20
            THEN 'medium_history'
        WHEN train_interactions >= 21
            THEN 'warm'
    END AS cohort,
    COUNT(*) AS item_count
FROM test_items
LEFT JOIN train_history USING (parent_asin)
GROUP BY cohort
ORDER BY
    CASE cohort
        WHEN 'true_cold' THEN 1
        WHEN 'low_history' THEN 2
        WHEN 'medium_history' THEN 3
        WHEN 'warm' THEN 4
    END
"""

test_item_cohorts = con.execute(query).df()

test_item_cohorts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cohort,item_count
0,true_cold,117203
1,low_history,34984
2,medium_history,32060
3,warm,62863


In [7]:
query = f"""
WITH train_history AS (
    SELECT
        parent_asin,
        COUNT(*) AS train_interactions
    FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
    WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) <= 2021
    GROUP BY parent_asin
),

test_interactions AS (
    SELECT
        parent_asin,
        COUNT(*) AS test_interactions
    FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
    WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) = 2023
    GROUP BY parent_asin
)

SELECT
    CASE
        WHEN COALESCE(train_interactions, 0) = 0
            THEN 'true_cold'
        WHEN train_interactions BETWEEN 1 AND 5
            THEN 'low_history'
        WHEN train_interactions BETWEEN 6 AND 20
            THEN 'medium_history'
        WHEN train_interactions >= 21
            THEN 'warm'
    END AS cohort,
    COUNT(*) AS item_count,
    SUM(test_interactions) AS test_interactions
FROM test_interactions
LEFT JOIN train_history USING (parent_asin)
GROUP BY cohort
ORDER BY
    CASE cohort
        WHEN 'true_cold' THEN 1
        WHEN 'low_history' THEN 2
        WHEN 'medium_history' THEN 3
        WHEN 'warm' THEN 4
    END
"""

test_cohort_stats = con.execute(query).df()

test_cohort_stats

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cohort,item_count,test_interactions
0,true_cold,117203,747151.0
1,low_history,34984,106820.0
2,medium_history,32060,129728.0
3,warm,62863,904999.0


In [8]:
query = f"""
SELECT DISTINCT parent_asin
FROM '{INTERACTIONS_PATH.as_posix()}/*.parquet'
WHERE EXTRACT(YEAR FROM to_timestamp(timestamp / 1000)) = 2023
"""

test_items = con.execute(query).df()

print(f"2023 test items: {len(test_items):,}")
test_items.head()

2023 test items: 247,110


,parent_asin
0,B07WV1MTVS
1,B07YKG5W32
2,B08XVF418R
3,B0C33L5T89
4,B09YRT8Y5T


In [9]:
TEST_ITEMS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "test_items_2023.parquet"
)

test_items.to_parquet(TEST_ITEMS_PATH, index=False)

print(TEST_ITEMS_PATH)

/home/henry-k/Documents/Projects/cold-start-recommender/data/processed/test_items_2023.parquet


In [10]:
import gzip
import json

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "meta_Electronics.jsonl.gz"
)

with gzip.open(METADATA_PATH, "rt", encoding="utf-8") as f:
    first_record = json.loads(next(f))

first_record.keys()

dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together'])

In [11]:
first_record

{'main_category': 'All Electronics',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'average_rating': 3.5,
 'rating_number': 6,
 'features': [],
 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.

In [13]:
import pandas as pd

sample_records = []

with gzip.open(METADATA_PATH, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10_000:
            break

        sample_records.append(json.loads(line))

metadata_sample = pd.DataFrame(sample_records)

metadata_sample.shape

(10000, 16)

In [14]:
metadata_sample.columns.tolist()

['main_category',
 'title',
 'average_rating',
 'rating_number',
 'features',
 'description',
 'price',
 'images',
 'videos',
 'store',
 'categories',
 'details',
 'parent_asin',
 'bought_together',
 'subtitle',
 'author']

In [15]:
metadata_sample[
    ["parent_asin", "title", "features", "description", "categories"]
].head()

,parent_asin,title,features,description,categories
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,[],[Teleporter V3 The “Teleporter V3” kit sets a ...,"[Electronics, Television & Video, Video Glasses]"
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],"[Electronics, Television & Video, Accessories,..."
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,[WARNING: Please IDENTIFY MODEL NUMBER on the ...,[],"[Electronics, Computers & Accessories, Laptop ..."
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,[☛NotoCity 22mm band is designed for Vivoactiv...,[],"[Electronics, Wearable Technology, Clips, Arm ..."
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,"[New Droid X Essentials Combo Pack, Exclusive ...",[all Genuine High Quality Motorola Made Access...,"[Electronics, Computers & Accessories, Compute..."


In [18]:
TEST_ITEMS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "test_items_2023.parquet"
)

print(TEST_ITEMS_PATH.exists())

True


In [19]:
METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "meta_Electronics.jsonl.gz"
)

metadata_path = METADATA_PATH.as_posix()
test_items_path = TEST_ITEMS_PATH.as_posix()

query = f"""
WITH metadata AS (
    SELECT parent_asin
    FROM read_json_auto(
        '{metadata_path}',
        format='newline_delimited'
    )
),

test_items AS (
    SELECT parent_asin
    FROM '{test_items_path}'
)

SELECT
    COUNT(*) AS test_items,
    COUNT(metadata.parent_asin) AS items_with_metadata,
    COUNT(*) - COUNT(metadata.parent_asin) AS items_without_metadata
FROM test_items
LEFT JOIN metadata
    USING (parent_asin)
"""

metadata_coverage = con.execute(query).df()

metadata_coverage

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,test_items,items_with_metadata,items_without_metadata
0,247110,247110,0
